In [ ]:
!pip -q install apify-client
import os, requests
from apify_client import ApifyClient

c = ApifyClient("apikeyhere")

usernames = [
    "name", "name2", "etc"
]

run = c.actor("apify/instagram-post-scraper").call(run_input={
    "username": usernames,
    "resultsLimit": 50,
})

os.makedirs("photos", exist_ok=True)
for p in c.dataset(run["defaultDatasetId"]).iterate_items():
    if p.get("type") == "Video":
        continue
    kids = [k for k in p.get("childPosts") or [] if k.get("type") != "Video"]
    urls = [k.get("displayUrl") for k in kids] or [p.get("displayUrl")]
    owner = p.get("ownerUsername", "unknown")
    os.makedirs(f"photos/{owner}", exist_ok=True)
    for i, u in enumerate(filter(None, urls)):
        r = requests.get(u, timeout=30)
        if r.ok:
            open(f"photos/{owner}/{p.get('shortCode','_')}_{i}.jpg", "wb").write(r.content)

!zip -qr photos.zip photos
from google.colab import files
files.download("photos.zip")